# 59 Three-way face detectability comparison

Run MediaPipe Face Detector on three mesh variants per subject, all built from the same pipeline run so the only factor that varies across columns is the operator applied: the original head-isolated CTF surface, the vertex-deletion output of the shipped pipeline, and the rejected noise-perturbation strawman (Cotangent-Laplacian neighbour pull + per-vertex Gaussian offsets, with a separate boundary-transition smoothing pass).

**Cohort.** Iterates over the eleven valid thesis subjects: the optode-cap cohort S1--S7 (Subject 16--22) and the bare-cap cohort S8--S11 (Subject 12--15). Subject 11 is excluded (unusable raw scan). The output CSV carries the shared `s_id` and `optode` columns so the LaTeX table can render the cohort split and a per-cohort summary directly.

Populates §4.4.4 (Table 4.7) of the thesis as the three-way extension of notebook 56. The colour contact sheet is restricted to S2 (Subject 17) under the data-sharing rule.

Output: `thesis_results_out/detectability_comparison_summary.csv`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_data import is_optode, s_id, SUBJECTS, load_raw, load_landmarks
from _thesis_pipeline import run_pipeline
from _validator_noise import noise_perturb_with_taper
from _validator_render import (
    YAWS, PITCHES,
    render_views, mediapipe_face_detector, mediapipe_detect,
)

import pandas as pd

import cedalion.dataclasses as cdc

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)
VIEW_ROOT = OUT_DIR / 'detectability_views_comparison'
VIEW_ROOT.mkdir(parents=True, exist_ok=True)

# Edit to redirect thesis figure exports; defaults to the local results dir.
THESIS_FIG_DIR = pathlib.Path('thesis_results_out')

_fd = mediapipe_face_detector()

In [ ]:
CONTACT_SUBJECT = 12  # only S2's color renders are produced (data-sharing rule)
COLOR_VIEW_ROOT = OUT_DIR / 'detectability_views_comparison_color'
COLOR_VIEW_ROOT.mkdir(parents=True, exist_ok=True)

rows = []
for n in SUBJECTS:
    print(f'\n=== Subject{n} ===', flush=True)
    raw = load_raw(n)
    lm  = load_landmarks(n)
    art = run_pipeline(raw, lm, subject=n)

    lm_ctf_xyz = art.landmarks_ctf.pint.dequantify().values

    # Noise variant: cotangent-Laplacian relaxation + per-vertex Gaussian
    # offsets, then a separate boundary-transition smoothing pass. Same code
    # path as the original anonymizer.NOISE method that produced the hero PNGs.
    noise_mesh = noise_perturb_with_taper(art.surface_ctf.mesh, art.mask, lm_ctf_xyz)
    surface_noise = cdc.TrimeshSurface(
        mesh=noise_mesh,
        crs=art.surface_ctf.crs,
        units=art.surface_ctf.units,
    )

    variants = {
        'original': art.surface_ctf,
        'delete':   art.surface_anon_ctf,
        'noise':    surface_noise,
    }

    for tag, surface in variants.items():
        # Detector sweep: grey, neutral lighting, identical across variants.
        sub_dir = VIEW_ROOT / f'subject{n}' / tag
        files = render_views(surface, sub_dir, tag, use_color=False)
        hits = 0
        max_conf = 0.0
        for yaw, pitch, fn in files:
            k, c = mediapipe_detect(fn, _fd)
            if k:
                hits += 1
            if c > max_conf:
                max_conf = c
        rows.append({
            's_id':     s_id(n),
            'subject':  n,
            'optode':   is_optode(n),
            'method':   tag,
            'n_views':  len(files),
            'hits':     hits,
            'max_conf': max_conf,
        })
        print(f'  {tag:10s}  hits={hits:2d}/{len(files):2d}  max_conf={max_conf:.3f}', flush=True)

        # Contact-sheet renders: color, only for the representative subject.
        if n == CONTACT_SUBJECT:
            color_dir = COLOR_VIEW_ROOT / f'subject{n}' / tag
            render_views(surface, color_dir, tag, use_color=True)

In [ ]:
df = pd.DataFrame(rows)
if len(df) and 's_id' in df.columns:
    df = df.iloc[df['s_id'].map(lambda s: int(s[1:])).argsort()].reset_index(drop=True)
csv = OUT_DIR / 'detectability_comparison_summary.csv'
df.to_csv(csv, index=False)
print(f'Wrote {csv}\n')

# Format like Table 4.4 of the thesis for visual comparison.
wide = df.pivot(index='subject', columns='method', values=['hits', 'max_conf'])
wide.columns = [f'{m}_{k}' for k, m in wide.columns]
wide = wide.reset_index().sort_values('subject').reset_index(drop=True)

# Re-label subjects to S1..S7 in the same fixed order used in the thesis.
S_MAP = {n: f'S{i+1}' for i, n in enumerate(sorted(df['subject'].unique()))}
wide.insert(0, 'ID', wide['subject'].map(S_MAP))
wide = wide.drop(columns=['subject'])
print(wide.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Thesis data-sharing rule: only Subject 17 (S2) appears in rendered figures.
TAGS  = ['original', 'delete', 'noise']
ROW_TITLES = ['Original', 'Vertex deletion', 'Noise perturbation']

fig, axes = plt.subplots(
    len(TAGS), len(YAWS), figsize=(2.4 * len(YAWS), 2.4 * len(TAGS))
)
for r, tag in enumerate(TAGS):
    for c, yaw in enumerate(YAWS):
        ax = axes[r, c]
        img = mpimg.imread(
            COLOR_VIEW_ROOT / f'subject{CONTACT_SUBJECT}' / tag /
            f'{tag}_yaw{yaw:+04d}_pitch+00.png'
        )
        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title(f'yaw {yaw:+d}°', fontsize=9)
        if c == 0:
            ax.set_ylabel(ROW_TITLES[r], fontsize=10)
fig.tight_layout()

out = THESIS_FIG_DIR / 'detectability_contact_S2.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Wrote {out}')